In [ ]:
# Carregando dataset MNIST a partir da OpenML
print("Baixando o dataset MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
print("Finalizou o processo de baixar o dataset MNIST!")

In [ ]:
# Separação entre features e target
X, y = mnist.data, mnist.target

In [ ]:
# Exploração da dimensionalidade das matrizes
print (f'========>> A matriz X possui {X.shape[0]} amostras, sendo {X.shape[1]} pixels por imagem')
print (f'========>> A matriz y é unidimensional e possui {y.shape[0]} rótulos / targets\n')
print (f'========>> Matriz de pixels de uma amostra:\n\n {X[10000]}')

In [ ]:
# Verificação da distribuição das classes e balanceamento do dataset
rotulos, freq_abs = np.unique(y, return_counts=True)
freq_perc = np.round(freq_abs / freq_abs.sum() * 100, decimals = 2)
print (f'Rótulos únicos: {rotulos}')
print (f'Frequências absolutas: {freq_abs}')
print (f'Frequências percentuais: {freq_perc}')

In [ ]:
# Averiguação dos tipos de dados
print (f'Tipos de dados da matriz X: {X.dtype}')
print (f'Tipos de dados da matriz y: {y.dtype}')

In [ ]:
# Alteração do tipo de dados da matriz de rótulos para inteiro
y = y.astype(np.uint8)

In [ ]:
# divisão estratificada dos dados em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=42)

In [ ]:
# escalonamento dos dados
scaler = MinMaxScaler()

X_train_escalonado = scaler.fit_transform(X_train)
X_test_escalonado = scaler.transform(X_test)

In [ ]:
''' Função que executa o Grid Search dos modelos.
    Recebe como parâmetro nome do modelo, modelo base, a grade de parâmetros para avaliar e conjunto de treino
    Retorna ao final o modelo com as melhores configurações
'''
def procura_melhores_parametros (nome_modelo, modelo, parametros, X_treino, y_treino):

    # Separa uma amostra de somente 9000 entradas para teste, para o Grid Search rodar mais rápido
    _, X_treino_gs, _, y_treino_gs = train_test_split(
    X_treino, y_treino, test_size=9000, stratify=y_treino, random_state=42
)
    # Configura o Grid Search com Validação Cruzada de 3 folds
    grid_search = GridSearchCV(
        estimator=modelo,
        param_grid=parametros,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,  # Usa todos os núcleos do processador em paralelo
        verbose=3,
    )

    # Executa a busca
    print(f"\nIniciando a busca de parâmetros otimizados do modelo {nome_modelo}...")
    inicio = time.time()
    grid_search.fit(X_treino_gs, y_treino_gs)
    fim = time.time()

    print(f"\nGrid Search concluído em {(fim - inicio)/60:.2f} minutos")

    print(f"Melhor combinação encontrada: {grid_search.best_params_}")
    print(f"Melhor Acurácia de Validação Cruzada: {grid_search.best_score_:.4f}")

    # Retorna o melhor classificador desta família
    return grid_search.best_estimator_